## 📝 Chap06-3. Make English Quiz
#### TTS로 영어 듣기 평가 문제 만들기
##### 영어문제 json으로 추출

In [2]:
import json
from openai import OpenAI
from dotenv import load_dotenv
import os

load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key=api_key)   

def make_quiz_from_image(image_path, n_trial=0, max_trial=3):
    if n_trial >= max_trial: # 최대 시도 회수에 도달하면 포기
        raise Exception("Failed to generate a quiz.")

    quiz_prompt = """
    제공된 이미지를 바탕으로, 다음과 같은 양식으로 퀴즈를 만들어주세요. 
    정답은 1~4 중 하나만 해당하도록 출제하세요.
    토익 리스닝 문제 스타일로 문제를 만들어주세요.
    아래는 예시입니다. 
    ----- 예시 -----

    Q: 다음 이미지에 대한 설명 중 옳지 않은 것은 무엇인가요?
    - (1) 왼쪽에서 첫번째 여성은 흰색 티셔츠를 입고 있습니다.
    - (2) 왼쪽에서 두번째 여성은 노란색 음료를 들고 있습니다.
    - (3) 왼쪽에서 세번째 여성은 선글라스를 착용하고 있습니다.
    - (4) 왼쪽에서 네번째 여성은 검은색 음료를 들고 있습니다.

    Listening: Which of the following descriptions of the image is incorrect?
    - (1) The first woman from the left is wearing a white T-shirt.
    - (2) The second woman from the left is holding a yellow drink.
    - (3) The third woman from the left is wearing sunglasses.
    - (4) The fourth woman from the left is holding a black drink.
        
    정답: (4) 왼쪽에서 네번째 여성은 검은색이 아니라 주황색 음료를 들고 있습니다.
    (주의: 정답은 1~4 중 하나만 선택되도록 출제하세요.)
    ======
    """

    messages = [
        {
            "role": "user",
            "content": [
                {"type": "text", "text": quiz_prompt},
                {
                    "type": "image_url",
                    "image_url": {
                        "url": image_path,
                    },
                },
            ],
        }
    ]

    try: 
        response = client.chat.completions.create(
            model="gpt-5.6-luna",
            messages=messages
        )
    except Exception as e:
        print("failed\n" + e)
        return make_quiz_from_image(image_path, n_trial+1)
    
    content = response.choices[0].message.content

    if "Listening:" in content:
        return content, True
    else:
        return make_quiz_from_image(image_path, n_trial+1)


q = make_quiz_from_image("https://images.unsplash.com/photo-1532635241-17e820acc59f?q=80&w=1115&auto=format&fit=crop&ixlib=rb-4.1.0&ixid=M3wxMjA3fDB8MHxwaG90by1wYWdlfHx8fGVufDB8fHx8fA%3D%3D")
print(q)


# 웹 이미지 URL 5개
urls = [
    "https://images.unsplash.com/photo-1511988617509-a57c8a288659?q=80&w=1171&auto=format&fit=crop&ixlib=rb-4.1.0&ixid=M3wxMjA3fDB8MHxwaG90by1wYWdlfHx8fGVufDB8fHx8fA%3D%3D",
    "https://images.unsplash.com/photo-1504022462188-88f023db97bf?q=80&w=1170&auto=format&fit=crop&ixlib=rb-4.1.0&ixid=M3wxMjA3fDB8MHxwaG90by1wYWdlfHx8fGVufDB8fHx8fA%3D%3D",
    "https://images.unsplash.com/photo-1517486808906-6ca8b3f04846?q=80&w=749&auto=format&fit=crop&ixlib=rb-4.1.0&ixid=M3wxMjA3fDB8MHxwaG90by1wYWdlfHx8fGVufDB8fHx8fA%3D%3D",
    "https://images.unsplash.com/photo-1549057446-9f5c6ac91a04?q=80&w=1634&auto=format&fit=crop&ixlib=rb-4.1.0&ixid=M3wxMjA3fDB8MHxwaG90by1wYWdlfHx8fGVufDB8fHx8fA%3D%3D",
    "https://images.unsplash.com/photo-1532675432006-329c6fed7045?q=80&w=627&auto=format&fit=crop&ixlib=rb-4.1.0&ixid=M3wxMjA3fDB8MHxwaG90by1wYWdlfHx8fGVufDB8fHx8fA%3D%3D"
]


txt = '' # 문제들을 계속 붙여 나가기 위해 빈 문자열 선언
eng_dict = []
no = 1 # 문제 번호를 위해 선언
for g in urls:
    q, is_suceed = make_quiz_from_image(g)

    if not is_suceed:
        continue


    divider = f'## 문제 {no}\n\n'
    print(divider)
    
    txt += divider
    # URL에서 파일명 추출(마크다운 레이블 용)
    label = f'문제 {no}'
    txt += f'![{label}]({g})\n\n'  # ③ 웹 이미지를 그대로 링크로 삽입

    # 문제 추가
    print(q)
    txt += q + '\n\n---------------------\n\n'
    # 마크다운 파일로 저장
    with open('data/images/image_quiz_eng.md', 'w', encoding='utf-8') as f:
        f.write(txt)

    # 영어 문제만 추출
    eng = q.split('Listening: ')[1].split('정답:')[0].strip()

    eng_dict.append({
        'no': no,
        'eng': eng,
        'img': label
    })

    # json 파일로 저장
    with open('data/images/image_quiz_eng.json', 'w', encoding='utf-8') as f:
        json.dump(eng_dict, f, ensure_ascii=False, indent=4)
    
    
    no += 1 # 문제 번호 증가

('Q: 다음 이미지에 대한 설명 중 옳지 않은 것은 무엇인가요?\n\n- (1) 왼쪽에서 첫 번째 여성은 흰색 티셔츠를 입고 어두운 색 음료를 들고 있습니다.\n- (2) 왼쪽에서 두 번째 여성은 검은색 상의를 입고 선글라스를 착용하고 있습니다.\n- (3) 왼쪽에서 세 번째 여성은 연한 파란색 셔츠를 입고 주황색 음료를 들고 있습니다.\n- (4) 왼쪽에서 네 번째 여성은 선글라스를 착용하고 빨간색 음료를 들고 있습니다.\n\nListening: Which of the following descriptions of the image is incorrect?\n\n- (1) The first woman from the left is wearing a white T-shirt and holding a dark-colored drink.\n- (2) The second woman from the left is wearing a black top and sunglasses.\n- (3) The third woman from the left is wearing a light blue shirt and holding an orange drink.\n- (4) The fourth woman from the left is wearing sunglasses and holding a red drink.\n\n정답: **(4)** 왼쪽에서 네 번째 여성은 빨간색 음료를 들고 있지만, 선글라스를 착용하고 있지는 않습니다.', True)
## 문제 1


Q: 다음 이미지에 대한 설명 중 옳지 않은 것은 무엇인가요?  
- (1) 왼쪽에서 첫 번째 남성은 안경을 쓰고 있습니다.  
- (2) 왼쪽에서 두 번째 여성은 선글라스를 착용하고 있습니다.  
- (3) 왼쪽에서 세 번째 남성은 목에 스카프를 두르고 있습니다.  
- (4) 오른쪽의 남성은 긴소매 셔츠를 입고 있습니다.  

Listening: Which of the following descriptions of the i

##### TTS 사용해 MP3 만들기

In [4]:
response = client.audio.speech.create(
    model="tts-1-hd",
    voice="nova",
    input="Hello world! This is a TTS test.",
)

response.write_to_file("hello_world.mp3")

# 재생
import IPython.display as ipd

ipd.Audio("hello_world.mp3")

In [5]:
import json

# json 파일 열기
with open('data/images/image_quiz_eng.json', 'r', encoding='utf-8') as f:
    eng_dict = json.load(f)

eng_dict

[{'no': 1,
  'eng': 'Which of the following descriptions of the image is incorrect?  \n- (1) The first man from the left is wearing glasses.  \n- (2) The second person from the left is wearing sunglasses.  \n- (3) The third man from the left is wearing a scarf around his neck.  \n- (4) The man on the right is wearing a long-sleeved shirt.',
  'img': '문제 1'},
 {'no': 2,
  'eng': 'Which of the following descriptions of the image is incorrect?  \n- (1) The man on the left is wearing glasses.  \n- (2) The woman in the center is wearing an orange sweater.  \n- (3) The woman on the right is wearing a black jacket over a red plaid shirt.  \n- (4) All three people are standing.',
  'img': '문제 2'},
 {'no': 3,
  'eng': 'Which of the following descriptions of the image is incorrect?  \n- (1) The first woman from the left is wearing a red top.  \n- (2) The second man from the left is wearing a blue shirt.  \n- (3) The third woman from the left is wearing a white top.  \n- (4) The first man from th

In [6]:
voices = ['alloy', 'ash', 'coral', 'echo', 'fable', 'onyx', 'nova', 'sage' , 'shimmer']

for q in eng_dict:
    no = q['no']
    quiz = q['eng']
    quiz = quiz.replace("- (1)", "- One.\t")
    quiz = quiz.replace("- (2)", "- Two.\t")
    quiz = quiz.replace("- (3)", "- Three.\t")
    quiz = quiz.replace("- (4)", "- Four.\t")    

    print(no, quiz)
    
    voice = voices[no % len(voices)] # 문제 개수를 목소리 개수로 나눈 나머지 값으로 선택  

    response = client.audio.speech.create(
        model="tts-1-hd",
        voice=voice,
        input=f'#{no}. {quiz}',
    )

    response.write_to_file(f"data/audio/{no}.mp3")

1 Which of the following descriptions of the image is incorrect?  
- One.	 The first man from the left is wearing glasses.  
- Two.	 The second person from the left is wearing sunglasses.  
- Three.	 The third man from the left is wearing a scarf around his neck.  
- Four.	 The man on the right is wearing a long-sleeved shirt.
2 Which of the following descriptions of the image is incorrect?  
- One.	 The man on the left is wearing glasses.  
- Two.	 The woman in the center is wearing an orange sweater.  
- Three.	 The woman on the right is wearing a black jacket over a red plaid shirt.  
- Four.	 All three people are standing.
3 Which of the following descriptions of the image is incorrect?  
- One.	 The first woman from the left is wearing a red top.  
- Two.	 The second man from the left is wearing a blue shirt.  
- Three.	 The third woman from the left is wearing a white top.  
- Four.	 The first man from the right is wearing sunglasses.
4 Which of the following descriptions of the 